# Análise Estatística — Desigualdade de Gênero no Mercado de Dados Brasileiro (2021–2024)


### Estrutura da análise

| Bloco | Análise | Objetivo |
|-------|---------|----------|
| 1 | Estatísticas descritivas | Caracterizar a amostra |
| 2 | Qui-quadrado + V de Cramér | Associação entre gênero e variáveis categoriais |
| 3 | Mann-Whitney U | Diferença salarial entre gêneros |
| 4 | Tendência temporal (regressão logística simples) | Mudança na representatividade feminina ao longo dos anos |
| 5 | Regressão logística múltipla | Preditores estruturais da desigualdade de gênero |
| 6 | Síntese e tabela de resultados | Resumo |

## 0. Setup e carregamento

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Estatística
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, pointbiserialr
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest

# Visualização
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.alpha': 0.3,
})

ALPHA = 0.05  # nível de significância

# ── Ordenações ────────────────────────────────────────────────────────────
FAIXA_SALARIAL_ORDEM = [
    'R$1k-2k','R$2k-3k','R$3k-4k','R$4k-6k','R$6k-8k',
    'R$8k-12k','R$12k-16k','R$16k-20k','R$20k-25k',
    'R$25k-30k','R$30k-40k','R$40k+',
]
NIVEL_ENSINO_ORDEM = [
    'Não tenho graduação formal','Estudante de Graduação',
    'Graduação/Bacharelado','Especialização Lato Sensu','Mestrado','Doutorado ou Phd',
]
NIVEL_SENIORIDADE_ORDEM = ['Júnior','Pleno','Sênior','Gestor']

# ── Carregamento ──────────────────────────────────────────────────────────
df = pd.read_excel('df_full.xlsx')
df['ano_pesquisa']   = df['ano_pesquisa'].astype(str)
df['faixa_salarial'] = pd.Categorical(df['faixa_salarial'], categories=FAIXA_SALARIAL_ORDEM, ordered=True)
df['nivel_ensino']   = pd.Categorical(df['nivel_ensino'],   categories=NIVEL_ENSINO_ORDEM,   ordered=True)
df['nivel']          = pd.Categorical(df['nivel'],          categories=NIVEL_SENIORIDADE_ORDEM, ordered=True)

# Codificação binária: Feminino = 1, Masculino = 0
df['genero_bin'] = (df['genero'] == 'Feminino').astype(int)

# Codificação numérica da faixa salarial (rank ordinal)
df['faixa_salarial_num'] = df['faixa_salarial'].cat.codes  # -1 = NaN
df['faixa_salarial_num'] = df['faixa_salarial_num'].replace(-1, np.nan)

# Ano como variável numérica para regressão temporal
df['ano_num'] = df['ano_pesquisa'].astype(int)
df['ano_centro'] = df['ano_num'] - 2021  # centraliza em 0 para melhor interpretação

print(f'N total: {len(df):,}')
print(f'N Feminino: {(df["genero"]=="Feminino").sum():,} ({(df["genero"]=="Feminino").mean()*100:.1f}%)')
print(f'N Masculino: {(df["genero"]=="Masculino").sum():,} ({(df["genero"]=="Masculino").mean()*100:.1f}%)')
print(f'Anos: {sorted(df["ano_pesquisa"].unique())}')

ModuleNotFoundError: No module named 'scipy'

---
## Bloco 1 — Estatísticas Descritivas

In [ ]:
# ── 1.1 Distribuição por gênero e ano ─────────────────────────────────────
tab_ano = df.groupby(['ano_pesquisa', 'genero']).size().unstack(fill_value=0)
tab_ano['Total'] = tab_ano.sum(axis=1)
tab_ano['% Feminino'] = (tab_ano['Feminino'] / tab_ano['Total'] * 100).round(2)
print('Distribuição por ano e gênero:')
print(tab_ano.to_string())

In [ ]:
# ── 1.2 Variáveis categóricas: frequência por gênero ──────────────────────
variaveis = {
    'Nível de Senioridade': 'nivel',
    'Faixa Salarial':       'faixa_salarial',
    'Nível de Ensino':      'nivel_ensino',
    'Cargo':                'cargo',
    'Modalidade':           'modalidade',
    'Situação':             'situacao',
    'Área de Formação':     'area_formacao',
    'Região':               'regiao',
}

for label, col in variaveis.items():
    sub = df.dropna(subset=[col])
    ct = sub.groupby([col, 'genero'], observed=True).size().unstack(fill_value=0)
    ct['Total'] = ct.sum(axis=1)
    ct['% Feminino'] = (ct.get('Feminino', 0) / ct['Total'] * 100).round(1)
    print(f'\n=== {label} ===')
    print(ct.to_string())

---
## Bloco 2 — Testes Qui-quadrado e Effect Size (V de Cramér)

**Hipótese nula (H₀):** Não há associação entre gênero e a variável categórica.  
**Hipótese alternativa (H₁):** Existe associação significativa entre gênero e a variável.  

**Interpretação do V de Cramér:**
- V < 0.10 → efeito negligenciável
- 0.10 ≤ V < 0.20 → efeito pequeno
- 0.20 ≤ V < 0.40 → efeito moderado
- V ≥ 0.40 → efeito forte

In [ ]:
def cramers_v(contingency_table):
    """Calcula o V de Cramér para uma tabela de contingência."""
    chi2, _, _, _ = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    k = min(contingency_table.shape) - 1
    return np.sqrt(chi2 / (n * k))


def teste_qui_quadrado(df, col, label):
    """Executa qui-quadrado e retorna dict com resultados."""
    sub = df.dropna(subset=[col, 'genero'])
    ct  = pd.crosstab(sub[col], sub['genero'])
    chi2, p, dof, expected = chi2_contingency(ct)
    v   = cramers_v(ct)

    # Verifica premissa: frequências esperadas ≥ 5
    pct_celulas_ok = (expected >= 5).mean()

    return {
        'Variável':         label,
        'N':                len(sub),
        'χ²':               round(chi2, 3),
        'gl':               dof,
        'p-valor':          round(p, 6),
        'Significativo':    'Sim' if p < ALPHA else 'Não',
        'V de Cramér':      round(v, 4),
        'Magnitude':        ('Forte' if v >= 0.4 else
                             'Moderado' if v >= 0.2 else
                             'Pequeno' if v >= 0.1 else 'Negligenciável'),
        '% células (exp≥5)': f'{pct_celulas_ok*100:.0f}%',
    }


resultados_chi2 = []
for label, col in variaveis.items():
    resultados_chi2.append(teste_qui_quadrado(df, col, label))

tab_chi2 = pd.DataFrame(resultados_chi2).set_index('Variável')
print('=== Resultados: Qui-quadrado + V de Cramér ===')
print(tab_chi2.to_string())

In [ ]:
# ── Visualização: V de Cramér por variável ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

tab_plot = tab_chi2.sort_values('V de Cramér', ascending=True)
cores_bar = ['#DC2626' if s == 'Sim' else '#9CA3AF' for s in tab_plot['Significativo']]

bars = ax.barh(tab_plot.index, tab_plot['V de Cramér'], color=cores_bar, alpha=0.85)
ax.axvline(0.10, color='#6B7280', linewidth=1, linestyle='--', label='Efeito pequeno (0.10)')
ax.axvline(0.20, color='#374151', linewidth=1, linestyle=':',  label='Efeito moderado (0.20)')

for bar, val in zip(bars, tab_plot['V de Cramér']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('V de Cramér (effect size)')
ax.set_title('Força da Associação entre Gênero e Variáveis do Mercado de Dados',
             fontsize=12, fontweight='bold', loc='left')
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig('fig_cramer_v.png', bbox_inches='tight', dpi=150)
plt.show()
print('Vermelho = associação estatisticamente significativa (p < 0.05)')

---
## Bloco 3 — Teste Mann-Whitney U: Desigualdade Salarial

Utilizado por ser **não paramétrico** — adequado para faixa salarial ordinal e para amostras com N desbalanceado (Masculino >> Feminino).  

**H₀:** A distribuição salarial de mulheres e homens é idêntica.  
**H₁:** Mulheres tendem a receber salários menores que homens.

**Effect size:** r = Z / √N (Cohen, 1992)

In [ ]:
def mann_whitney_salarial(df, grupo_col='genero', sal_col='faixa_salarial_num', label='Geral'):
    sub = df.dropna(subset=[sal_col, grupo_col])
    masc = sub[sub[grupo_col] == 'Masculino'][sal_col]
    fem  = sub[sub[grupo_col] == 'Feminino'][sal_col]

    U, p = mannwhitneyu(masc, fem, alternative='greater')  # H1: masc > fem
    n = len(masc) + len(fem)
    z = stats.norm.ppf(1 - p)
    r = abs(z) / np.sqrt(n)  # effect size

    return {
        'Grupo':              label,
        'N Masculino':        len(masc),
        'N Feminino':         len(fem),
        'Mediana Masc (rank)': masc.median(),
        'Mediana Fem (rank)':  fem.median(),
        'Faixa modal Masc':   FAIXA_SALARIAL_ORDEM[int(masc.mode()[0])] if len(masc) > 0 else 'N/A',
        'Faixa modal Fem':    FAIXA_SALARIAL_ORDEM[int(fem.mode()[0])] if len(fem) > 0 else 'N/A',
        'U':                  round(U, 0),
        'p-valor':            round(p, 6),
        'Significativo':      'Sim' if p < ALPHA else 'Não',
        'r (effect size)':    round(r, 4),
        'Magnitude r':        ('Grande' if r >= 0.5 else
                               'Médio' if r >= 0.3 else
                               'Pequeno' if r >= 0.1 else 'Negligenciável'),
    }


# Teste geral
res_mw = [mann_whitney_salarial(df, label='Brasil — Geral')]

# Teste por ano
for ano in ['2021', '2022', '2023', '2024']:
    res_mw.append(mann_whitney_salarial(df[df['ano_pesquisa'] == ano], label=f'Brasil — {ano}'))

# Teste por região
regioes = df['regiao'].dropna().unique()
for reg in sorted(regioes):
    res_mw.append(mann_whitney_salarial(df[df['regiao'] == reg], label=f'Região: {reg}'))

tab_mw = pd.DataFrame(res_mw).set_index('Grupo')
print('=== Mann-Whitney U: Diferença Salarial por Gênero ===')
print(tab_mw.to_string())

In [ ]:
# ── Visualização: distribuição salarial por gênero ─────────────────────────
sub_sal = df.dropna(subset=['faixa_salarial'])
sal_pct = (
    sub_sal.groupby(['faixa_salarial', 'genero'], observed=True)
    .size().unstack(fill_value=0)
    .apply(lambda x: x / x.sum() * 100)
)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(FAIXA_SALARIAL_ORDEM))
w = 0.38
for i, genero in enumerate(['Feminino', 'Masculino']):
    cor = '#FF7F50' if genero == 'Feminino' else '#1E90FF'
    vals = sal_pct.reindex(FAIXA_SALARIAL_ORDEM)[genero] if genero in sal_pct.columns else [0]*len(x)
    ax.bar(x + (i - 0.5)*w, vals, w, color=cor, alpha=0.88, label=genero)

ax.set_xticks(x)
ax.set_xticklabels(FAIXA_SALARIAL_ORDEM, rotation=35, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylabel('% dentro do gênero')
ax.set_title('Distribuição Salarial por Gênero — % dentro de cada gênero',
             fontsize=12, fontweight='bold', loc='left')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('fig_salario_genero.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Bloco 4 — Tendência Temporal: Regressão Logística Simples

**Pergunta:** A proporção de mulheres no mercado de dados mudou significativamente entre 2021 e 2024?  

**Modelo:** `P(Feminino) = logit⁻¹(β₀ + β₁ × ano_centro)`  

- `ano_centro` = ano − 2021 (0, 1, 2, 3)  
- **β₁ > 0** → proporção feminina crescendo; **β₁ < 0** → decrescendo  
- Interpretação do OR: a cada ano adicional, a chance de o respondente ser mulher muda pelo fator OR.

In [ ]:
def regressao_temporal(df_sub, label='Brasil'):
    sub = df_sub.dropna(subset=['genero_bin', 'ano_centro'])
    X   = sm.add_constant(sub['ano_centro'])
    modelo = sm.Logit(sub['genero_bin'], X).fit(disp=False)

    coef  = modelo.params['ano_centro']
    p     = modelo.pvalues['ano_centro']
    OR    = np.exp(coef)
    ci_lo = np.exp(modelo.conf_int().loc['ano_centro', 0])
    ci_hi = np.exp(modelo.conf_int().loc['ano_centro', 1])

    return {
        'Grupo':          label,
        'β (log-odds/ano)': round(coef, 4),
        'OR':             round(OR, 4),
        'IC 95% OR':      f'[{ci_lo:.3f}, {ci_hi:.3f}]',
        'p-valor':        round(p, 6),
        'Significativo':  'Sim' if p < ALPHA else 'Não',
        'Tendência':      ('Crescente' if coef > 0 and p < ALPHA else
                           'Decrescente' if coef < 0 and p < ALPHA else 'Estável'),
    }


res_temp = [regressao_temporal(df, 'Brasil — Geral')]

for reg in sorted(df['regiao'].dropna().unique()):
    res_temp.append(regressao_temporal(df[df['regiao'] == reg], f'Região: {reg}'))

tab_temp = pd.DataFrame(res_temp).set_index('Grupo')
print('=== Tendência Temporal da Representatividade Feminina ===')
print(tab_temp.to_string())

In [ ]:
# ── Visualização: proporção feminina observada + curva ajustada ────────────
pct_obs = df.groupby('ano_num').apply(
    lambda x: (x['genero'] == 'Feminino').mean() * 100
).reset_index(name='pct_fem')

# Curva logística ajustada
sub_fit = df.dropna(subset=['genero_bin', 'ano_centro'])
X_fit   = sm.add_constant(sub_fit['ano_centro'])
mod_fit = sm.Logit(sub_fit['genero_bin'], X_fit).fit(disp=False)
anos_seq = np.linspace(0, 3, 100)
X_pred   = sm.add_constant(anos_seq)
y_pred   = mod_fit.predict(X_pred) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(pct_obs['ano_num'], pct_obs['pct_fem'],
           color='#FF7F50', s=80, zorder=5, label='Proporção observada')
ax.plot(anos_seq + 2021, y_pred,
        color='#DC2626', linewidth=2, linestyle='--', label='Curva logística ajustada')

for _, row in pct_obs.iterrows():
    ax.annotate(f"{row['pct_fem']:.1f}%",
                (row['ano_num'], row['pct_fem']),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=9, fontweight='bold', color='#DC2626')

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel('Ano')
ax.set_ylabel('% Respondentes Feminino')
ax.set_title('Tendência Temporal da Representatividade Feminina (2021–2024)',
             fontsize=12, fontweight='bold', loc='left')
ax.legend(frameon=False)
ax.set_ylim(0, 35)
plt.tight_layout()
plt.savefig('fig_tendencia_temporal.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Bloco 5 — Regressão Logística Múltipla

**Objetivo:** Identificar quais fatores estruturais predizem a probabilidade de o respondente ser mulher, controlando simultaneamente por cargo, nível, faixa salarial, região e ano.

**Variável dependente:** `genero_bin` (1 = Feminino, 0 = Masculino)  
**Preditores:** `faixa_salarial_num` (ordinal), `nivel`, `cargo`, `regiao`, `ano_centro`

**Interpretação dos OR:**
- OR < 1 → o preditor está associado a **menor** probabilidade de ser mulher
- OR > 1 → o preditor está associado a **maior** probabilidade de ser mulher
- OR controla pelos demais preditores no modelo

In [ ]:
# Prepara dados para regressão múltipla
cols_modelo = ['genero_bin', 'faixa_salarial_num', 'nivel', 'cargo', 'regiao', 'ano_centro']
df_modelo = df[cols_modelo].dropna().copy()

# Transforma categorias em dummies (drop_first=True para evitar multicolinearidade)
df_dummies = pd.get_dummies(
    df_modelo,
    columns=['nivel', 'cargo', 'regiao'],
    drop_first=True
)

# Converte booleanos para int
bool_cols = df_dummies.select_dtypes(include='bool').columns
df_dummies[bool_cols] = df_dummies[bool_cols].astype(int)

y = df_dummies['genero_bin']
X = sm.add_constant(df_dummies.drop(columns=['genero_bin']))

modelo_mult = sm.Logit(y, X).fit(disp=False, maxiter=200)
print(modelo_mult.summary())

In [ ]:
# ── Tabela de Odds Ratios com IC 95% ──────────────────────────────────────
params = modelo_mult.params
conf   = modelo_mult.conf_int()
pvals  = modelo_mult.pvalues

tab_or = pd.DataFrame({
    'OR':         np.exp(params),
    'IC 95% inf': np.exp(conf[0]),
    'IC 95% sup': np.exp(conf[1]),
    'p-valor':    pvals,
    'Sig.':       pvals.apply(lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))
}).drop(index='const')

tab_or = tab_or.round(4)
print('=== Odds Ratios — Regressão Logística Múltipla ===')
print(f'N = {len(y):,} | Pseudo-R² (McFadden) = {modelo_mult.prsquared:.4f}')
print(f'AIC = {modelo_mult.aic:.1f} | Log-likelihood = {modelo_mult.llf:.1f}')
print()
print(tab_or.to_string())

In [ ]:
# ── Forest plot dos Odds Ratios ────────────────────────────────────────────
sig_mask = tab_or['p-valor'] < ALPHA
tab_or_sig = tab_or[sig_mask].sort_values('OR')

fig, ax = plt.subplots(figsize=(9, max(4, len(tab_or_sig) * 0.45)))

cores = ['#DC2626' if v < 1 else '#2563EB' for v in tab_or_sig['OR']]
y_pos = np.arange(len(tab_or_sig))

ax.scatter(tab_or_sig['OR'], y_pos, color=cores, s=60, zorder=5)
for i, (_, row) in enumerate(tab_or_sig.iterrows()):
    ax.plot([row['IC 95% inf'], row['IC 95% sup']], [i, i],
            color=cores[i], linewidth=1.5, alpha=0.7)

ax.axvline(1.0, color='black', linewidth=1.2, linestyle='--', alpha=0.5, label='OR = 1 (sem efeito)')
ax.set_yticks(y_pos)
ax.set_yticklabels(tab_or_sig.index, fontsize=8)
ax.set_xlabel('Odds Ratio (IC 95%)')
ax.set_title('Forest Plot — Preditores Significativos da Probabilidade de Ser Mulher\n'
             'Regressão Logística Múltipla (p < 0.05)',
             fontsize=11, fontweight='bold', loc='left')
ax.annotate('← menor prob. feminina | maior prob. feminina →',
            xy=(0.5, -0.1), xycoords='axes fraction',
            ha='center', fontsize=8, color='#6B7280')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig('fig_forest_plot.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Bloco 6 — Síntese dos Resultados

In [ ]:
print('=' * 70)
print('SÍNTESE DOS RESULTADOS ESTATÍSTICOS')
print('=' * 70)

print('\n[1] ASSOCIAÇÃO GÊNERO × VARIÁVEIS (Qui-quadrado + V de Cramér)')
print('-' * 60)
for _, row in tab_chi2.sort_values('V de Cramér', ascending=False).iterrows():
    sig = '✓' if row['Significativo'] == 'Sim' else '✗'
    print(f"  {sig} {row.name:<30} V={row['V de Cramér']:.3f} ({row['Magnitude']}) | p={row['p-valor']:.4f}")

print('\n[2] DESIGUALDADE SALARIAL (Mann-Whitney U)')
print('-' * 60)
row_mw = tab_mw.loc['Brasil — Geral']
print(f"  U={row_mw['U']:.0f} | p={row_mw['p-valor']:.6f} | r={row_mw['r (effect size)']:.3f} ({row_mw['Magnitude r']})")
print(f"  Faixa modal Masc: {row_mw['Faixa modal Masc']} | Faixa modal Fem: {row_mw['Faixa modal Fem']}")

print('\n[3] TENDÊNCIA TEMPORAL (Regressão Logística Simples)')
print('-' * 60)
row_t = tab_temp.loc['Brasil — Geral']
print(f"  β={row_t['β (log-odds/ano)']:.4f} | OR={row_t['OR']:.4f} | IC95%={row_t['IC 95% OR']} | p={row_t['p-valor']:.4f}")
print(f"  Tendência: {row_t['Tendência']}")

print('\n[4] PREDITORES ESTRUTURAIS (Regressão Logística Múltipla)')
print('-' * 60)
print(f"  N={len(y):,} | Pseudo-R²={modelo_mult.prsquared:.4f} | AIC={modelo_mult.aic:.1f}")
sig_preds = tab_or[tab_or['p-valor'] < ALPHA].sort_values('OR')
print(f"  Preditores significativos: {len(sig_preds)}/{len(tab_or)}")
print('  Top 5 que REDUZEM chance de ser mulher (OR < 1):')
for nome, row in sig_preds[sig_preds['OR'] < 1].head(5).iterrows():
    print(f"    {nome:<40} OR={row['OR']:.3f} | p={row['p-valor']:.4f}")
print('  Top 5 que AUMENTAM chance de ser mulher (OR > 1):')
for nome, row in sig_preds[sig_preds['OR'] > 1].tail(5).iterrows():
    print(f"    {nome:<40} OR={row['OR']:.3f} | p={row['p-valor']:.4f}")

In [ ]:
# ── Exporta tabelas para Excel (anexo do relatório) ────────────────────────
with pd.ExcelWriter('resultados_estatisticos.xlsx') as writer:
    tab_chi2.to_excel(writer, sheet_name='Qui-quadrado e Cramer V')
    tab_mw.to_excel(writer, sheet_name='Mann-Whitney Salarial')
    tab_temp.to_excel(writer, sheet_name='Tendência Temporal')
    tab_or.to_excel(writer, sheet_name='Regressão Logística Múltipla')
print('Tabelas exportadas: resultados_estatisticos.xlsx')